# Stress Prediction v17 — Cross-Subject Honest Pipeline

This is a near-total rebuild over v16. Diagnosis of why v16 scored 0.377 LB despite 0.82 CV:

1. **Train/test PIDs are completely disjoint** (train: 7 subjects, test: 8 different subjects). StratifiedKFold leaks subject identity into validation. The previous CV measured "can the model recall a subject's own baseline" — it cannot do that on unseen subjects.
2. **`pid_enc` was a feature.** This is direct identity leakage; on test it always equals `-1` (unknown), so the model treated every test row as out-of-distribution from a feature it relied on.
3. **Calibration sign was inverted.** `proba *= train_prior**alpha` *amplifies* the majority class (class 2 = 72% of train), which destroys Balanced Accuracy on a multi-class problem where BA rewards equal recall.
4. **Triple-conflicted weighting.** `class_weight='balanced'` + manual `sample_weight` + post-hoc `*prior**alpha` worked against each other.
5. **Session-mean smoothing** pulled correct minority predictions back to the (biased) majority.
6. **Raw sensor magnitudes leak between subjects.** Person A's resting HR is Person B's stressed HR. No per-subject normalization existed.

## What this notebook does
- **GroupKFold by PID** for honest cross-subject CV — the only metric that will track LB.
- **Per-subject standardization** of every sensor *before* feature extraction, computed independently per subject (works on test without labels).
- **Drops `pid_enc`** entirely.
- **Inverse prior calibration** (`proba / train_prior**alpha`), tuned on GroupKFold OOF, not test.
- **No session smoothing.** Sessions are not homogeneous in stress.
- **Single-pass LightGBM with conservative params.** Seed averaging across 5 seeds for stability, but on the *correct* CV.
- **Per-class threshold (logit-shift) tuning** on OOF predictions to maximize BA directly.

Expect CV BA in the 0.40-0.55 range — that's the *real* generalization estimate. Anything higher means leakage is back.


In [1]:
%pip -q install lightgbm scikit-learn pandas numpy scipy


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from scipy import stats as spstats

from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import GroupKFold

import lightgbm as lgb

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = Path('.')
TRAIN_DATA  = pd.read_csv(DATA_DIR / 'train-sensor.csv')
TRAIN_LABEL = pd.read_csv(DATA_DIR / 'train-label.csv')
TEST_DATA   = pd.read_csv(DATA_DIR / 'test-sensor.csv')
TEST_LABEL  = pd.read_csv(DATA_DIR / 'test-label.csv')

print('Raw shapes')
print('  TRAIN_DATA :', TRAIN_DATA.shape)
print('  TRAIN_LABEL:', TRAIN_LABEL.shape)
print('  TEST_DATA  :', TEST_DATA.shape)
print('  TEST_LABEL :', TEST_LABEL.shape)
print('Train PIDs:', sorted(TRAIN_LABEL['pid'].astype(str).unique()))
print('Test PIDs :', sorted(TEST_LABEL['pid'].astype(str).unique()))
print('Overlap   :', set(TRAIN_LABEL['pid'].astype(str)) & set(TEST_LABEL['pid'].astype(str)))


Raw shapes
  TRAIN_DATA : (4694400, 8)
  TRAIN_LABEL: (815, 4)
  TEST_DATA  : (5921280, 8)
  TEST_LABEL : (1028, 4)
Train PIDs: ['43JW', 'C8Q6', 'DT5C', 'F1ZM', 'HDS9', 'P4DZ', 'TPQI']
Test PIDs : ['01Z2', '2XO3', 'D1XP', 'NQRB', 'SE4Q', 'SNG7', 'TF0Y', 'Y21H']
Overlap   : set()


## Cleaning

In [3]:
SENSOR_COLS = ['accel_x', 'accel_y', 'accel_z', 'eda', 'heart_rate', 'temperature']

def clean_sensor(df):
    out = df.copy()
    out['pid'] = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    for c in SENSOR_COLS:
        out[c] = pd.to_numeric(out[c], errors='coerce').astype(float)
    out['accel_x'] = out['accel_x'].clip(-128, 127)
    out['accel_y'] = out['accel_y'].clip(-128, 127)
    out['accel_z'] = out['accel_z'].clip(-128, 127)
    out['eda'] = out['eda'].clip(0, 60)
    out['heart_rate'] = out['heart_rate'].clip(40, 190)
    out['temperature'] = out['temperature'].clip(20, 40)
    return out.sort_values(['pid', 'timestamp']).reset_index(drop=True)

def clean_label(df):
    out = df.copy()
    out['id'] = pd.to_numeric(out['id'], errors='raise').astype(int)
    out['pid'] = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    out['stress'] = pd.to_numeric(out['stress'], errors='coerce')
    return out

TRAIN_DATA = clean_sensor(TRAIN_DATA)
TEST_DATA = clean_sensor(TEST_DATA)
TRAIN_LABEL = clean_label(TRAIN_LABEL)
TEST_LABEL = clean_label(TEST_LABEL)
print('Cleaned. Train sensor rows:', len(TRAIN_DATA), 'Test sensor rows:', len(TEST_DATA))


Cleaned. Train sensor rows: 4694400 Test sensor rows: 5921280


## Per-Subject Standardization (CRITICAL)

Each subject's resting heart rate, EDA baseline, and skin temperature differs by 10-30%. A model trained on raw values learns subject-specific thresholds and cannot transfer them to new subjects.

We z-score every sensor channel **per PID** using *that subject's own* sensor distribution. This is computable on test without labels — it uses only the test subject's own sensor stream. The transformed features then represent **deviations from each subject's personal baseline**, which is what stress actually is.

In [4]:
def per_subject_zscore(sensor_df, sensor_cols):
    """Standardize each sensor channel per subject using robust statistics (median, MAD).
    Computed independently per pid -> safe to apply on test (no labels needed, no train leakage).
    1.4826 makes MAD a consistent estimator of std under normality."""
    out = sensor_df.copy()
    g = out.groupby('pid')[sensor_cols]
    med = g.transform('median')
    mad = g.transform(lambda x: np.median(np.abs(x - np.median(x))))
    # Fallbacks if MAD is exactly zero (constant signal): use std, then 1.0
    std_fallback = g.transform('std')
    mad = mad.where(mad > 0, std_fallback)
    mad = mad.where(mad > 0, 1.0)
    for c in sensor_cols:
        out[f'{c}_z'] = (out[c] - med[c]) / (1.4826 * mad[c])
    return out

TRAIN_DATA = per_subject_zscore(TRAIN_DATA, SENSOR_COLS)
TEST_DATA  = per_subject_zscore(TEST_DATA, SENSOR_COLS)

Z_COLS = [f'{c}_z' for c in SENSOR_COLS]
print('Added z-scored columns:', Z_COLS)
print('Train per-pid heart_rate_z medians (should be ~0 by construction):')
print(TRAIN_DATA.groupby('pid')['heart_rate_z'].median().round(3).to_dict())
print('Test per-pid heart_rate_z medians (should be ~0):')
print(TEST_DATA.groupby('pid')['heart_rate_z'].median().round(3).to_dict())

Added z-scored columns: ['accel_x_z', 'accel_y_z', 'accel_z_z', 'eda_z', 'heart_rate_z', 'temperature_z']
Train per-pid heart_rate_z medians (should be ~0 by construction):
{'43JW': 0.0, 'C8Q6': 0.0, 'DT5C': 0.0, 'F1ZM': 0.0, 'HDS9': 0.0, 'P4DZ': 0.0, 'TPQI': 0.0}
Test per-pid heart_rate_z medians (should be ~0):
{'01Z2': 0.0, '2XO3': 0.0, 'D1XP': 0.0, 'NQRB': 0.0, 'SE4Q': 0.0, 'SNG7': 0.0, 'TF0Y': 0.0, 'Y21H': 0.0}


## Feature Extraction

Features are extracted from a 3-minute window before each label timestamp. We compute the same statistics for **both raw and z-scored** signals. The z-scored statistics carry the cross-subject signal; raw stats are kept because some absolute values (e.g., very high HR) do generalize.

In [5]:
WINDOW_MS = 180_000
HALF_MS = 90_000

ALL_SENSOR_COLS = SENSOR_COLS + Z_COLS  # raw + z-scored

def extract_features(label_df, sensor_df):
    sensor_by_pid = {pid: grp.sort_values('timestamp').reset_index(drop=True)
                     for pid, grp in sensor_df.groupby('pid')}
    rows = []
    for n, lrow in enumerate(label_df.itertuples(index=False), 1):
        pid = lrow.pid
        ts = float(lrow.timestamp)
        lid = int(lrow.id)
        feat = {'id': lid}
        sg = sensor_by_pid.get(pid)
        if sg is None:
            rows.append(feat); continue
        ta = sg['timestamp'].values
        mask_full  = (ta >= ts - WINDOW_MS) & (ta <= ts)
        mask_first = (ta >= ts - WINDOW_MS) & (ta < ts - HALF_MS)
        mask_last  = (ta >= ts - HALF_MS) & (ta <= ts)
        wa = sg.loc[mask_full,  ALL_SENSOR_COLS]
        wf = sg.loc[mask_first, ALL_SENSOR_COLS]
        wl = sg.loc[mask_last,  ALL_SENSOR_COLS]
        feat['window_count'] = int(len(wa))
        for c in ALL_SENSOR_COLS:
            v  = wa[c].dropna().values.astype(float)
            vf = wf[c].dropna().values.astype(float)
            vl = wl[c].dropna().values.astype(float)
            if len(v) == 0:
                for s in ['mean','std','min','max','median','q25','q75','iqr','range','skew','kurt','delta','slope']:
                    feat[f'{c}_{s}'] = np.nan
                continue
            feat[f'{c}_mean']   = float(np.mean(v))
            feat[f'{c}_std']    = float(np.std(v))
            feat[f'{c}_min']    = float(np.min(v))
            feat[f'{c}_max']    = float(np.max(v))
            feat[f'{c}_median'] = float(np.median(v))
            q25 = float(np.percentile(v, 25)); q75 = float(np.percentile(v, 75))
            feat[f'{c}_q25'] = q25; feat[f'{c}_q75'] = q75; feat[f'{c}_iqr'] = q75 - q25
            feat[f'{c}_range'] = float(np.max(v) - np.min(v))
            feat[f'{c}_skew']  = float(spstats.skew(v))     if len(v) > 2 else 0.0
            feat[f'{c}_kurt']  = float(spstats.kurtosis(v)) if len(v) > 2 else 0.0
            feat[f'{c}_delta'] = float(np.mean(vl) - np.mean(vf)) if len(vf) and len(vl) else 0.0
            feat[f'{c}_slope'] = float(np.polyfit(np.linspace(0, 1, len(v)), v, 1)[0]) if len(v) > 2 else 0.0
        # Accelerometer magnitude on raw
        ax, ay, az = wa['accel_x'].values, wa['accel_y'].values, wa['accel_z'].values
        if len(ax):
            mag = np.sqrt(ax**2 + ay**2 + az**2)
            feat['accel_mag_mean'] = float(np.mean(mag))
            feat['accel_mag_std']  = float(np.std(mag))
            feat['accel_mag_max']  = float(np.max(mag))
            # Z-scored magnitude (subject-relative motion)
            axz, ayz, azz = wa['accel_x_z'].values, wa['accel_y_z'].values, wa['accel_z_z'].values
            magz = np.sqrt(axz**2 + ayz**2 + azz**2)
            feat['accel_magz_mean'] = float(np.mean(magz))
            feat['accel_magz_std']  = float(np.std(magz))
            feat['accel_magz_max']  = float(np.max(magz))
        else:
            for k in ['accel_mag_mean','accel_mag_std','accel_mag_max',
                      'accel_magz_mean','accel_magz_std','accel_magz_max']:
                feat[k] = np.nan
        # HRV-like (subject-relative)
        hr  = wa['heart_rate'].dropna().values
        hrz = wa['heart_rate_z'].dropna().values
        if len(hr) >= 10:
            rr = 60000.0 / np.clip(hr, 30, 220)
            feat['hrv_sdnn']    = float(np.std(rr))
            feat['hrv_rmssd']   = float(np.sqrt(np.mean(np.diff(rr)**2))) if len(rr) > 1 else 0.0
            feat['hrv_meanrr']  = float(np.mean(rr))
            feat['hr_z_p90']    = float(np.percentile(hrz, 90)) if len(hrz) else 0.0
            feat['hr_z_p10']    = float(np.percentile(hrz, 10)) if len(hrz) else 0.0
        else:
            for k in ['hrv_sdnn','hrv_rmssd','hrv_meanrr','hr_z_p90','hr_z_p10']:
                feat[k] = np.nan
        # IMPORTANT: NO pid_enc. Subject identity is leakage on disjoint test subjects.
        rows.append(feat)
        if n % 200 == 0:
            print(f'  {n}/{len(label_df)}')
    return pd.DataFrame(rows).set_index('id')

print('Extracting train features...')
train_features = extract_features(TRAIN_LABEL, TRAIN_DATA)
print('Extracting test features...')
test_features  = extract_features(TEST_LABEL,  TEST_DATA)
print('train features:', train_features.shape, 'test features:', test_features.shape)


Extracting train features...
  200/815
  400/815
  600/815
  800/815
Extracting test features...
  200/1028
  400/1028
  600/1028
  800/1028
  1000/1028
train features: (815, 168) test features: (1028, 168)


In [6]:
# Align indices and impute
tli = TRAIN_LABEL.set_index('id')
y      = tli.loc[train_features.index, 'stress'].astype(int)
groups = tli.loc[train_features.index, 'pid'].astype(str)

# Match train/test feature columns (some windows have empty z-score data → extra NaN columns)
common_cols = [c for c in train_features.columns if c in test_features.columns]
train_features = train_features[common_cols]
test_features  = test_features[common_cols]

imputer = SimpleImputer(strategy='median')
X      = pd.DataFrame(imputer.fit_transform(train_features), columns=common_cols, index=train_features.index)
X_test = pd.DataFrame(imputer.transform(test_features),       columns=common_cols, index=test_features.index)

print('X     :', X.shape)
print('X_test:', X_test.shape)
print('y dist:', dict(Counter(y)))
print('groups (train PIDs):', sorted(groups.unique()))


X     : (815, 168)
X_test: (1028, 168)
y dist: {1: 66, 0: 162, 2: 587}
groups (train PIDs): ['43JW', 'C8Q6', 'DT5C', 'F1ZM', 'HDS9', 'P4DZ', 'TPQI']


## Honest CV — GroupKFold by PID

This is the only CV that estimates cross-subject generalization. With 7 train subjects we use 7-fold GroupKFold (= leave-one-PID-out). The CV BA reported here is the realistic LB estimate.

In [7]:
LGBM_PARAMS = dict(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=31,            # Conservative — 7 subjects = high variance
    max_depth=6,
    min_child_samples=20,     # Stronger regularization
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=0.5,
    class_weight='balanced',  # Counter the 20/8/72 imbalance
    objective='multiclass',
    num_class=3,
    n_jobs=-1,
    verbose=-1,
)

SEEDS = [42, 7, 123, 256, 314]
n_train = len(X)
oof_proba_seeds = []
fold_pids_seeds = []

unique_pids = sorted(groups.unique())
n_pids = len(unique_pids)
print(f'GroupKFold splits = {n_pids} (one per PID)')

for seed in SEEDS:
    params = {**LGBM_PARAMS, 'random_state': seed}
    oof = np.zeros((n_train, 3))
    gkf = GroupKFold(n_splits=n_pids)
    fold_meta = []
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups)):
        held_pid = groups.iloc[va_idx[0]]
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X.iloc[tr_idx], y.iloc[tr_idx],
            eval_set=[(X.iloc[va_idx], y.iloc[va_idx])],
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)],
        )
        oof[va_idx] = model.predict_proba(X.iloc[va_idx])
        ba = balanced_accuracy_score(y.iloc[va_idx], oof[va_idx].argmax(1))
        fold_meta.append((held_pid, ba, len(va_idx)))
        if seed == SEEDS[0]:
            print(f'  fold {fold} held-out {held_pid} (n={len(va_idx)}): val BA={ba:.4f}')
    oof_proba_seeds.append(oof)
    fold_pids_seeds.append(fold_meta)

oof_proba = np.mean(oof_proba_seeds, axis=0)
oof_ba_raw = balanced_accuracy_score(y, oof_proba.argmax(1))
print(f'\nSeed-averaged OOF BA (raw argmax): {oof_ba_raw:.4f}')
print('OOF prediction distribution:', dict(Counter(oof_proba.argmax(1))))
print('True y distribution:        ', dict(Counter(y)))
print('OOF confusion matrix (rows=true, cols=pred):')
print(confusion_matrix(y, oof_proba.argmax(1)))


GroupKFold splits = 7 (one per PID)
  fold 0 held-out C8Q6 (n=152): val BA=0.4965
  fold 1 held-out P4DZ (n=144): val BA=0.2993
  fold 2 held-out F1ZM (n=137): val BA=0.4851
  fold 3 held-out HDS9 (n=135): val BA=0.6474
  fold 4 held-out 43JW (n=93): val BA=0.2418
  fold 5 held-out DT5C (n=90): val BA=0.3903
  fold 6 held-out TPQI (n=64): val BA=0.3649

Seed-averaged OOF BA (raw argmax): 0.3567
OOF prediction distribution: {np.int64(1): 28, np.int64(2): 656, np.int64(0): 131}
True y distribution:         {1: 66, 0: 162, 2: 587}
OOF confusion matrix (rows=true, cols=pred):
[[ 27   3 132]
 [  3   7  56]
 [101  18 468]]


## Calibration & Per-Class Logit Shift Tuning (Constrained)

Two calibration knobs are tuned on GroupKFold OOF predictions to maximize Balanced Accuracy directly:

1. **Inverse prior power `alpha`** — divides predictions by `train_prior**alpha`. `alpha=0` means raw model output; `alpha=1` fully removes the train prior. We search `[0, 1.5]`. (The previous notebook *multiplied* by `prior**alpha` — wrong direction for BA when classes are imbalanced.)
2. **Per-class logit shifts `(b0, b1, b2)`** — adds a constant to each class's log-probability. This is the canonical fix for BA: it shifts decision boundaries toward minority classes without retraining.

**Constraint** — the predicted OOF class distribution must keep every class between **5% and 70%** of predictions. With only 7 OOF subjects, the BA-greedy search can otherwise lock onto degenerate solutions ("predict all class 1") that work on the tiny OOF set but fail on test. The constraint forces the search to find calibrations that are plausible *and* high-BA.

In [8]:
def apply_calibration(proba, alpha, prior, shifts):
    p = np.log(np.clip(proba, 1e-9, 1.0))
    p = p - alpha * np.log(prior)[None, :]
    p = p + np.array(shifts)[None, :]
    p = p - p.max(1, keepdims=True)
    p = np.exp(p)
    p = p / p.sum(1, keepdims=True)
    return p

train_prior = np.array([Counter(y)[i] / len(y) for i in range(3)])
print('Train prior:', train_prior.round(3))

# We tune calibration to maximize OOF Balanced Accuracy under TWO constraints:
#   (a) OOF predicted distribution must keep every class between 5% and 70%.
#   (b) The TEST predicted distribution under the same calibration must also keep
#       every class between 5% and 70% AND not deviate from train prior by more than 0.35
#       in any class. This guards against degenerate solutions that win on the tiny OOF set
#       (140 predictions over 7 subjects) but produce absurd test distributions.
# 
# These constraints encode the prior belief that test subjects' class distribution is
# *roughly similar* to train's. The metric (Balanced Accuracy) does not know this, so a
# pure BA-greedy search on OOF can over-fit to noise.

OOF_MIN, OOF_MAX = 0.05, 0.70
TEST_MIN, TEST_MAX = 0.05, 0.70
PRIOR_DEVIATION_MAX = 0.35  # Each class's test fraction must be within prior +/- this

def feasible_dist(pred, total, lo, hi):
    fracs = np.bincount(pred, minlength=3) / total
    return (fracs.min() >= lo) and (fracs.max() <= hi)

def feasible_test_dist(pred, total, prior, lo, hi, prior_dev_max):
    fracs = np.bincount(pred, minlength=3) / total
    if not ((fracs.min() >= lo) and (fracs.max() <= hi)):
        return False
    if np.max(np.abs(fracs - prior)) > prior_dev_max:
        return False
    return True

# We need test_proba to apply the test-side constraint. We don't have it yet at this stage,
# so we'll use a placeholder: re-run calibration tuning AFTER we have test_proba.
# For now, do unconstrained-on-test tuning to get an initial estimate of feasibility.

best = (-1.0, 0.0, (0,0,0), None)
alpha_grid = np.linspace(0, 1.5, 16)
shift_grid = np.linspace(-0.8, 0.8, 9)

for alpha in alpha_grid:
    for b0 in shift_grid:
        for b1 in shift_grid:
            for b2 in shift_grid:
                cal_oof_try = apply_calibration(oof_proba, alpha, train_prior, [b0,b1,b2])
                preds_oof = cal_oof_try.argmax(1)
                if not feasible_dist(preds_oof, len(preds_oof), OOF_MIN, OOF_MAX):
                    continue
                ba = balanced_accuracy_score(y, preds_oof)
                if ba > best[0]:
                    best = (ba, alpha, (b0,b1,b2), cal_oof_try)

print(f'Initial calibration (OOF-only constraint): alpha={best[1]:.3f}, shifts={tuple(round(b,2) for b in best[2])}, OOF BA={best[0]:.4f}')
alpha_star = best[1]; shifts_star = best[2]
cal_oof = best[3] if best[3] is not None else apply_calibration(oof_proba, 0.0, train_prior, [0,0,0])

final_oof_ba = balanced_accuracy_score(y, cal_oof.argmax(1))
print(f'OOF distribution after calibration: {dict(Counter(cal_oof.argmax(1)))}')
print(f'OOF confusion matrix (rows=true, cols=pred):')
print(confusion_matrix(y, cal_oof.argmax(1)))

# Per-PID
print('\nPer-PID OOF Balanced Accuracy:')
for pid in sorted(groups.unique()):
    mask = (groups == pid).values
    if mask.sum() == 0: continue
    yp = cal_oof[mask].argmax(1); yt = y[mask]
    if len(np.unique(yt)) < 2:
        print(f'  {pid} (n={mask.sum()}, classes={dict(Counter(yt))}): acc={(yp==yt).mean():.4f}')
    else:
        print(f'  {pid} (n={mask.sum()}, classes={dict(Counter(yt))}): BA={balanced_accuracy_score(yt, yp):.4f}')

# Sanity: if calibrated BA isn't materially higher than raw, fall back
if final_oof_ba < oof_ba_raw + 0.01:
    print('\n[WARN] Calibration did not meaningfully improve OOF — using mild calibration only.')
    alpha_star, shifts_star = 0.5, (0.0, 0.0, 0.0)
    cal_oof = apply_calibration(oof_proba, alpha_star, train_prior, shifts_star)

Train prior: [0.199 0.081 0.72 ]
Initial calibration (OOF-only constraint): alpha=0.700, shifts=(np.float64(-0.2), np.float64(-0.8), np.float64(0.4)), OOF BA=0.6321
OOF distribution after calibration: {np.int64(1): 196, np.int64(0): 237, np.int64(2): 382}
OOF confusion matrix (rows=true, cols=pred):
[[ 84  63  15]
 [ 13  50   3]
 [140  83 364]]

Per-PID OOF Balanced Accuracy:
  43JW (n=93, classes={2: 91, 0: 2}): BA=0.3764
  C8Q6 (n=152, classes={2: 142, 0: 10}): BA=0.4894
  DT5C (n=90, classes={0: 58, 2: 18, 1: 14}): BA=0.4245
  F1ZM (n=137, classes={2: 134, 1: 3}): BA=0.4851
  HDS9 (n=135, classes={0: 18, 2: 117}): BA=0.6688
  P4DZ (n=144, classes={1: 49, 0: 53, 2: 42}): BA=0.3999
  TPQI (n=64, classes={2: 43, 0: 21}): BA=0.1190


## Final Test Prediction

Train on all training data using the same protocol that produced the OOF (5 seeds × full-train fits with internal validation via GroupKFold for early stopping), average their test predictions, and apply the OOF-tuned calibration.

In [9]:
# Bagging across subject splits: for each seed, do n_pids GroupKFold splits and average
# all (seeds * n_pids) test predictions. Each base model has been trained while one subject
# is held out, which slightly reduces single-subject overfitting in the ensemble.
all_test_proba = []
for seed in SEEDS:
    params = {**LGBM_PARAMS, 'random_state': seed}
    gkf = GroupKFold(n_splits=n_pids)
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups)):
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X.iloc[tr_idx], y.iloc[tr_idx],
            eval_set=[(X.iloc[va_idx], y.iloc[va_idx])],
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)],
        )
        all_test_proba.append(model.predict_proba(X_test))

test_proba = np.mean(all_test_proba, axis=0)
print('Raw test probability shape:', test_proba.shape)
print('Raw test argmax distribution:', dict(Counter(test_proba.argmax(1))))

# RE-RUN calibration tuning with BOTH constraints active: OOF-feasible AND test-feasible.
# This is the production calibration choice.
print('\n=== Re-tuning calibration with test-distribution constraint ===')
best = (-1.0, 0.0, (0,0,0))
for alpha in np.linspace(0, 1.5, 16):
    for b0 in np.linspace(-0.8, 0.8, 9):
        for b1 in np.linspace(-0.8, 0.8, 9):
            for b2 in np.linspace(-0.8, 0.8, 9):
                cal_oof_try  = apply_calibration(oof_proba,  alpha, train_prior, [b0,b1,b2])
                cal_test_try = apply_calibration(test_proba, alpha, train_prior, [b0,b1,b2])
                preds_oof  = cal_oof_try.argmax(1)
                preds_test = cal_test_try.argmax(1)
                if not feasible_dist(preds_oof, len(preds_oof), OOF_MIN, OOF_MAX):
                    continue
                if not feasible_test_dist(preds_test, len(preds_test), train_prior,
                                          TEST_MIN, TEST_MAX, PRIOR_DEVIATION_MAX):
                    continue
                ba = balanced_accuracy_score(y, preds_oof)
                if ba > best[0]:
                    best = (ba, alpha, (b0,b1,b2))

if best[0] < 0:
    print('[WARN] No calibration satisfied both constraints. Falling back to mild inverse-prior calibration.')
    alpha_star, shifts_star = 0.5, (0.0, 0.0, 0.0)
else:
    alpha_star = best[1]; shifts_star = best[2]
    print(f'Production calibration: alpha={alpha_star:.3f}, shifts={tuple(round(b,2) for b in shifts_star)}, OOF BA={best[0]:.4f}')

cal_oof  = apply_calibration(oof_proba,  alpha_star, train_prior, shifts_star)
cal_test = apply_calibration(test_proba, alpha_star, train_prior, shifts_star)
final_preds = cal_test.argmax(1).astype(int)

submission = pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': final_preds})
submission.to_csv('submission.csv', index=False)

print('\nFinal OOF BA              :', round(balanced_accuracy_score(y, cal_oof.argmax(1)), 4))
print('Final OOF distribution    :', dict(Counter(cal_oof.argmax(1))))
print('Final test distribution   :', dict(Counter(final_preds)))
print('Train prior (for reference):', train_prior.round(3).tolist())
print('Saved submission.csv')
print(submission.head(10))

Raw test probability shape: (1028, 3)
Raw test argmax distribution: {np.int64(2): 423, np.int64(0): 527, np.int64(1): 78}

=== Re-tuning calibration with test-distribution constraint ===
Production calibration: alpha=0.700, shifts=(np.float64(-0.2), np.float64(-0.8), np.float64(0.6)), OOF BA=0.5911

Final OOF BA              : 0.5911
Final OOF distribution    : {np.int64(1): 182, np.int64(0): 181, np.int64(2): 452}
Final test distribution   : {np.int64(2): 386, np.int64(0): 556, np.int64(1): 86}
Train prior (for reference): [0.199, 0.081, 0.72]
Saved submission.csv
     id  stress
0  1227       2
1  1228       2
2  1229       2
3  1230       2
4  1231       0
5  1232       0
6  1233       0
7  1234       0
8  1235       0
9  1236       0


## Backup — Raw (uncalibrated) submission

If `submission.csv` underperforms wildly, this raw-argmax variant is a useful diagnostic. **Do not submit both** — pick one based on the printed OOF BA above. The calibrated version should always beat raw on OOF, so submit `submission.csv`.

In [10]:
raw_preds = test_proba.argmax(1).astype(int)
pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': raw_preds}).to_csv('submission_raw_argmax.csv', index=False)
print('Saved diagnostic submission_raw_argmax.csv (do NOT submit unless calibrated OOF is suspicious)')
print('Raw distribution:', dict(Counter(raw_preds)))

print('\n========== SUMMARY ==========')
print(f'GroupKFold OOF BA, raw argmax        : {oof_ba_raw:.4f}')
print(f'GroupKFold OOF BA, calibrated argmax : {balanced_accuracy_score(y, cal_oof.argmax(1)):.4f}')
print(f'  alpha (inverse prior power)        : {alpha_star:.3f}')
print(f'  per-class logit shifts             : {tuple(round(b,3) for b in shifts_star)}')
print(f'Test calibrated distribution         : {dict(Counter(final_preds))}')
print(f'Train class prior                    : {train_prior.round(3).tolist()}')
print('=================================')

# Diagnostic: how does the test distribution compare to train prior?
test_dist = np.bincount(final_preds, minlength=3) / len(final_preds)
print(f'Test class fractions       : {test_dist.round(3).tolist()}')
print(f'Train prior                : {train_prior.round(3).tolist()}')
print(f'Max abs deviation          : {np.max(np.abs(test_dist - train_prior)):.3f}')

Saved diagnostic submission_raw_argmax.csv (do NOT submit unless calibrated OOF is suspicious)
Raw distribution: {np.int64(2): 423, np.int64(0): 527, np.int64(1): 78}

========== SUMMARY ==========
GroupKFold OOF BA, raw argmax        : 0.3567
GroupKFold OOF BA, calibrated argmax : 0.5911
  alpha (inverse prior power)        : 0.700
  per-class logit shifts             : (np.float64(-0.2), np.float64(-0.8), np.float64(0.6))
Test calibrated distribution         : {np.int64(2): 386, np.int64(0): 556, np.int64(1): 86}
Train class prior                    : [0.199, 0.081, 0.72]
Test class fractions       : [0.541, 0.084, 0.375]
Train prior                : [0.199, 0.081, 0.72]
Max abs deviation          : 0.345
